<a href="https://colab.research.google.com/github/profpoojabatra/Artificial-Intelligence/blob/main/Schedulestudent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted! Plans will save to: /content/drive/MyDrive/study_plans/")

Mounted at /content/drive
✅ Google Drive mounted! Plans will save to: /content/drive/MyDrive/study_plans/


In [2]:
import math
import random
import os
import glob

# ── SAVE DIRECTORY (Google Drive) ──────────────────
SAVE_DIR = "/content/drive/MyDrive/study_plans"

# ── MOTIVATIONAL QUOTES ────────────────────────────
QUOTES = [
    "🌟 'The secret of getting ahead is getting started.' — Mark Twain",
    "💡 'It always seems impossible until it's done.' — Nelson Mandela",
    "🔥 'Push yourself, because no one else is going to do it for you.'",
    "🚀 'Dream big. Start small. Act now.' — Robin Sharma",
    "🏆 'Success is the sum of small efforts repeated day in and day out.' — R. Collier",
    "⚡ 'Don't watch the clock; do what it does — keep going.' — Sam Levenson",
    "🌈 'The harder you work for something, the greater you'll feel when you achieve it.'",
    "🎯 'Focus on being productive instead of busy.' — Tim Ferriss",
    "💪 'You don't have to be great to start, but you have to start to be great.'",
    "🌙 'Believe you can and you're halfway there.' — Theodore Roosevelt",
]

REST_QUOTES = [
    "😴 'Sleep is the best meditation.' — Dalai Lama",
    "🧘 'Almost everything will work again if you unplug it for a few minutes — including you.'",
    "💤 'Rest is not idleness. It is the key to a sharper mind tomorrow.'",
    "🌿 'Your body is telling you something. Listen to it and be kind to yourself.'",
]

# ── SUBJECT EMOJIS ─────────────────────────────────
SUBJECT_EMOJIS = {
    "math": "📐", "mathematics": "📐", "maths": "📐",
    "physics": "⚛️", "chemistry": "🧪", "biology": "🧬",
    "english": "📖", "history": "🏛️", "geography": "🌍",
    "computer": "💻", "cs": "💻", "python": "🐍",
    "ai": "🤖", "ml": "🧠", "data": "📊",
    "economics": "💹", "accounts": "📒", "commerce": "💰",
    "hindi": "📜", "science": "🔬", "social": "🌐",
    "dbms": "🗄️", "networks": "🌐", "os": "🖥️",
}

print("✅ Cell 2 done — constants loaded!")

✅ Cell 2 done — constants loaded!


In [3]:
def get_emoji(subject_name):
    name = subject_name.lower().strip()
    for key, emoji in SUBJECT_EMOJIS.items():
        if key in name:
            return emoji
    return "📚"

def print_divider(char="─", width=58):
    print(char * width)

def print_section(title):
    print()
    print_divider()
    print(f"  {title}")
    print_divider()

def minutes_to_time(minutes):
    minutes = minutes % (24 * 60)
    h = minutes // 60
    m = minutes % 60
    period = "AM" if h < 12 else "PM"
    display_h = h if h <= 12 else h - 12
    if display_h == 0:
        display_h = 12
    return f"{display_h:02d}:{m:02d} {period}"

def get_fatigue_profile(fatigue):
    if fatigue <= 3:
        return {
            "study_mins": 25, "break_mins": 5, "cycle": 30,
            "label": "🟢 Fresh Mode",
            "advice": "You're energised! Standard Pomodoro — full power ahead. 💪",
            "cap_poms": None,
        }
    elif fatigue <= 6:
        return {
            "study_mins": 20, "break_mins": 7, "cycle": 27,
            "label": "🟡 Moderate Mode",
            "advice": "Slightly tired — shorter sessions with more rest built in.",
            "cap_poms": None,
        }
    elif fatigue <= 8:
        return {
            "study_mins": 15, "break_mins": 10, "cycle": 25,
            "label": "🟠 Tired Mode",
            "advice": "Sessions cut to 15 min with longer breaks.\n  💡 Max 2 Pomodoros per subject. Prioritise the most important ones.",
            "cap_poms": 2,
        }
    else:
        return {
            "study_mins": 10, "break_mins": 10, "cycle": 20,
            "label": "🔴 Rest Mode",
            "advice": "Very high fatigue! Minimal plan created.\n  ⚠️ Please sleep first — a rested brain learns 2× better!",
            "cap_poms": 1,
        }

print("✅ Cell 3 done — helper functions ready!")

✅ Cell 3 done — helper functions ready!


In [4]:
def build_pomodoro_schedule(subject_name, hours_available, start_slot, profile):
    emoji       = get_emoji(subject_name)
    study_mins  = profile["study_mins"]
    break_mins  = profile["break_mins"]
    cycle       = profile["cycle"]
    cap_poms    = profile["cap_poms"]

    total_minutes = int(hours_available * 60)
    max_possible  = total_minutes // cycle
    pomodoros     = max_possible if cap_poms is None else min(max_possible, cap_poms)
    leftover      = total_minutes - (pomodoros * cycle)

    blocks  = []
    current = start_slot

    for p in range(pomodoros):
        study_start = minutes_to_time(current)
        study_end   = minutes_to_time(current + study_mins)
        break_end   = minutes_to_time(current + cycle)

        blocks.append({
            "type": "study", "subject": subject_name, "emoji": emoji,
            "start": study_start, "end": study_end,
            "label": f"Pomodoro #{p+1}  ({study_mins} min)",
        })
        blocks.append({
            "type": "break",
            "start": study_end, "end": break_end,
            "label": f"☕ Break  ({break_mins} min)",
        })
        current += cycle

    # Mini leftover session
    if leftover >= 10 and (cap_poms is None or pomodoros < cap_poms):
        mini = min(leftover, study_mins)
        blocks.append({
            "type": "study", "subject": subject_name, "emoji": emoji,
            "start": minutes_to_time(current),
            "end":   minutes_to_time(current + mini),
            "label": f"Mini Session  ({mini} min)",
        })
        current += mini

    return blocks, current

print("✅ Cell 4 done — schedule builder ready!")

✅ Cell 4 done — schedule builder ready!


In [5]:
def ensure_save_dir():
    if not os.path.exists(SAVE_DIR):
        os.makedirs(SAVE_DIR)

def save_plan_to_file(name, date, fatigue, profile, subjects_data, all_blocks, quote):
    ensure_save_dir()
    safe_date = date.replace(" ", "_").replace("/", "-")
    safe_name = name.replace(" ", "_")
    filename  = os.path.join(SAVE_DIR, f"study_plan_{safe_date}_{safe_name}.txt")

    study_blocks = [b for b in all_blocks if b["type"] == "study"]
    break_blocks  = [b for b in all_blocks if b["type"] == "break"]
    W = 60

    lines = []
    lines.append("=" * W)
    lines.append(f"  STUDY PLAN FOR : {name.upper()}")
    lines.append(f"  DATE           : {date}")
    lines.append(f"  FATIGUE LEVEL  : {fatigue}/10  |  {profile['label']}")
    lines.append(f"  SESSION MODE   : {profile['study_mins']} min study / {profile['break_mins']} min break")
    lines.append("=" * W)
    lines.append(f"  Subjects    : {len(subjects_data)}")
    lines.append(f"  Study blocks: {len(study_blocks)}")
    lines.append(f"  Breaks      : {len(break_blocks)}")
    lines.append("-" * W)
    lines.append("  DETAILED SCHEDULE")
    lines.append("-" * W)

    prev_subject = None
    for block in all_blocks:
        if block["type"] == "study":
            if block["subject"] != prev_subject:
                lines.append(f"\n  {block['emoji']}  {block['subject'].upper()}")
                prev_subject = block["subject"]
            lines.append(f"     [STUDY]  {block['start']} --> {block['end']}  |  {block['label']}")
        else:
            lines.append(f"     [BREAK]  {block['start']} --> {block['end']}  |  {block['label']}")

    lines.append("-" * W)
    lines.append("  SUBJECT BREAKDOWN")
    lines.append("-" * W)
    for subject, hours in subjects_data:
        emoji = get_emoji(subject)
        poms  = sum(1 for b in all_blocks if b["type"] == "study"
                    and b["subject"] == subject and "Mini" not in b["label"])
        lines.append(f"  {emoji}  {subject:<22}  {hours:.1f} hr  -->  {poms} Pomodoro(s)")

    lines.append("-" * W)
    lines.append("  TODAY'S QUOTE")
    lines.append("-" * W)
    lines.append(f"  {quote}")
    lines.append("=" * W)

    with open(filename, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    return filename

def list_saved_plans():
    ensure_save_dir()
    return sorted(glob.glob(os.path.join(SAVE_DIR, "study_plan_*.txt")), reverse=True)

def display_saved_plan(filepath):
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            content = f.read()
        print_divider("═")
        print("  📂  SAVED PLAN")
        print_divider("═")
        print(content)
        print_divider("═")
    except FileNotFoundError:
        print(f"  ⚠️  File not found: {filepath}")

def offer_plan_review():
    saved = list_saved_plans()
    if not saved:
        print("  ℹ️  No saved plans found — let's create your first one!\n")
        return False

    print(f"  💾  {len(saved)} saved plan(s) found!\n")
    print("  [1] View most recent saved plan")
    print("  [2] Choose from all saved plans")
    print("  [3] Skip — create today's plan now")

    while True:
        choice = input("\n  Your choice (1/2/3): ").strip()
        if choice == "1":
            display_saved_plan(saved[0])
            cont = input("\n  Create today's new plan now? (y/n): ").strip().lower()
            return cont != "y"
        elif choice == "2":
            print("\n  📂  All saved plans:")
            for i, f in enumerate(saved, 1):
                print(f"  [{i}]  {os.path.basename(f)}")
            while True:
                try:
                    idx = int(input(f"\n  Enter number (1–{len(saved)}): ").strip())
                    if 1 <= idx <= len(saved):
                        display_saved_plan(saved[idx - 1])
                        cont = input("\n  Create today's new plan now? (y/n): ").strip().lower()
                        return cont != "y"
                    print(f"  ⚠️  Enter a number between 1 and {len(saved)}.")
                except ValueError:
                    print("  ⚠️  Please enter a valid number.")
        elif choice == "3":
            return False
        else:
            print("  ⚠️  Please enter 1, 2, or 3.")

print("✅ Cell 5 done — save & review functions ready!")

✅ Cell 5 done — save & review functions ready!


In [6]:
def print_schedule(name, date, fatigue, profile, subjects_data, all_blocks, quote):
    study_blocks = [b for b in all_blocks if b["type"] == "study"]
    break_blocks  = [b for b in all_blocks if b["type"] == "break"]

    print()
    print("╔" + "═" * 56 + "╗")
    print(f"║  📋  STUDY PLAN FOR {name.upper():<36}║")
    print(f"║  📅  {date:<51}║")
    print(f"║  {profile['label']:<55}║")
    print("╚" + "═" * 56 + "╝")

    print(f"\n  📊  Summary:")
    print(f"     • Fatigue level  : {fatigue}/10")
    print(f"     • Session length : {profile['study_mins']} min study / {profile['break_mins']} min break")
    print(f"     • Subjects       : {len(subjects_data)}")
    print(f"     • Study blocks   : {len(study_blocks)}")
    print(f"     • Breaks         : {len(break_blocks)}")

    print()
    print_divider()
    print("  💡  FATIGUE ADVICE:")
    for line in profile["advice"].split("\n"):
        print(f"  {line.strip()}")
    print_divider()

    print_section("📅  YOUR DETAILED SCHEDULE")
    prev_subject = None
    for block in all_blocks:
        if block["type"] == "study":
            subj = block["subject"]
            if subj != prev_subject:
                print(f"\n  {block['emoji']}  {subj.upper()}")
                prev_subject = subj
            print(f"     🟢  {block['start']} → {block['end']}  │  {block['label']}")
        else:
            print(f"     ☕  {block['start']} → {block['end']}  │  {block['label']}")

    print()
    print_divider()
    print_section("📌  SUBJECT BREAKDOWN")
    for subject, hours in subjects_data:
        emoji    = get_emoji(subject)
        poms     = sum(1 for b in all_blocks if b["type"] == "study"
                       and b["subject"] == subject and "Mini" not in b["label"])
        has_mini = any(b["type"] == "study" and b["subject"] == subject
                       and "Mini" in b["label"] for b in all_blocks)
        skipped  = int(hours * 60) // profile["cycle"] - poms if profile["cap_poms"] else 0
        tags     = ("+ mini " if has_mini else "") + (f"⚠️ {skipped} block(s) skipped" if skipped > 0 else "")
        print(f"  {emoji}  {subject:<22}  {hours:.1f} hr  →  {poms} Pomodoro(s)  {tags}")

    print_divider()
    print_section(f"⏱️  POMODORO RULES  ({profile['study_mins']} min study / {profile['break_mins']} min break)")
    print(f"  🟢  {profile['study_mins']} min  →  Study hard. No phone. Full focus.")
    print(f"  ☕   {profile['break_mins']} min  →  Stand up. Stretch. Water. Breathe.")
    if fatigue >= 7:
        print("  🛌  After each subject — close your eyes for 2 min. Rest counts!")
    print("  🏆  Every 4 Pomodoros → Take a 20 min long break!")
    print_divider()

    print()
    print("  ✨  MOTIVATION FOR TODAY:")
    print(f"  {quote}")
    print()
    print_divider("═")
    if fatigue >= 9:
        print(f"  🌙  Rest well, {name}. Even a small plan is better than none. 🤍")
    elif fatigue >= 7:
        print(f"  💛  Gentle session, {name}! You're doing great by even showing up. 🌟")
    else:
        print(f"  🎓  You've got this, {name}! One Pomodoro at a time. 💪")
    print_divider("═")
    print()

print("✅ Cell 6 done — display function ready!")

✅ Cell 6 done — display function ready!


In [7]:
def main():
    print("\n" + "═" * 58)
    print("║  🎓  AI STUDY PLANNER AGENT v2.0  ·  Pomodoro+  🎓  ║")
    print("═" * 58)
    print("\n  Hello! I'm your AI Study Planner 🤖")
    print("  Fatigue-aware · Auto-saves to Google Drive · Plan memory\n")

    # Step 0 — Review saved plans
    stop = offer_plan_review()
    if stop:
        print("\n  👋  See you next time! Keep studying smart. 🎓\n")
        return

    # Step 1 — Basic info
    print()
    print_divider()
    while True:
        name = input("👤  What's your name? ").strip()
        if name:
            break
        print("    ⚠️  Please enter your name.")

    print(f"\n  Nice to meet you, {name}! 🎉")

    while True:
        date = input("📅  Today's date (e.g. 27 June 2026): ").strip()
        if date:
            break
        print("    ⚠️  Please enter the date.")

    # Step 2 — Fatigue
    print("\n😴  How TIRED are you right now?")
    print("    1  = Wide awake and energised")
    print("    5  = Somewhat tired but manageable")
    print("    7  = Noticeably tired, hard to focus")
    print("    10 = Exhausted, barely keeping eyes open")
    while True:
        try:
            fatigue = int(input("\n    Your fatigue level (1–10): ").strip())
            if 1 <= fatigue <= 10:
                break
            print("    ⚠️  Enter a number between 1 and 10.")
        except ValueError:
            print("    ⚠️  Please enter a number.")

    profile = get_fatigue_profile(fatigue)
    print(f"\n  Mode selected →  {profile['label']}")
    print(f"  {profile['advice'].splitlines()[0]}")

    # Step 3 — Start time
    print("\n⏰  What time do you want to START? (24-hour format)")
    print("    e.g. 9 for 9 AM, 14 for 2 PM")
    while True:
        try:
            hour = int(input("    Start hour (0–23): ").strip())
            if 0 <= hour <= 23:
                start_slot = hour * 60
                break
            print("    ⚠️  Enter a valid hour (0–23).")
        except ValueError:
            print("    ⚠️  Please enter a number.")

    # Step 4 — Subjects
    while True:
        try:
            subject_count = int(input("\n📚  How many subjects today? ").strip())
            if 1 <= subject_count <= 10:
                break
            print("    ⚠️  Enter a number between 1 and 10.")
        except ValueError:
            print("    ⚠️  Please enter a valid number.")

    if fatigue >= 7 and subject_count > 2:
        print(f"\n  ⚠️  High fatigue ({fatigue}/10) + {subject_count} subjects detected.")
        print("  🔶  Pomodoro blocks will be capped. Focus on your top 2 subjects!\n")

    print()
    print_divider()
    print(f"  📚  Tell me about your {subject_count} subject(s):")
    print_divider()

    subjects_data = []
    for i in range(1, subject_count + 1):
        print(f"\n  ── Subject {i} ──")
        while True:
            sname = input("  📖  Subject name: ").strip()
            if sname:
                break
            print("      ⚠️  Please enter a subject name.")
        while True:
            try:
                hours = float(input(f"  ⏳  Hours for {sname}: ").strip())
                if 0.25 <= hours <= 12:
                    subjects_data.append((sname, hours))
                    break
                print("      ⚠️  Enter hours between 0.25 and 12.")
            except ValueError:
                print("      ⚠️  Please enter a valid number.")

    # Step 5 — Build schedule
    print("\n  ⚙️  Building your fatigue-adjusted schedule...")
    all_blocks   = []
    current_time = start_slot

    for subject_name, hours in subjects_data:
        blocks, current_time = build_pomodoro_schedule(
            subject_name, hours, current_time, profile
        )
        all_blocks.extend(blocks)

    if all_blocks and all_blocks[-1]["type"] == "break":
        all_blocks.pop()

    if not any(b["type"] == "study" for b in all_blocks):
        print("\n  🔴  Fatigue too high and hours too low — no blocks generated.")
        print("  Please REST first! 🛌")
        print(f"\n  {random.choice(REST_QUOTES)}\n")
        return

    quote = random.choice(REST_QUOTES if fatigue >= 8 else QUOTES)

    # Step 6 — Print plan
    print_schedule(name, date, fatigue, profile, subjects_data, all_blocks, quote)

    # Step 7 — Save
    filename = save_plan_to_file(
        name, date, fatigue, profile, subjects_data, all_blocks, quote
    )
    print(f"  💾  Plan saved → {filename}")
    print(f"  📂  Find all plans in Google Drive → study_plans/ folder\n")

print("✅ Cell 7 done — main() is ready!")

✅ Cell 7 done — main() is ready!


In [8]:
main()


══════════════════════════════════════════════════════════
║  🎓  AI STUDY PLANNER AGENT v2.0  ·  Pomodoro+  🎓  ║
══════════════════════════════════════════════════════════

  Hello! I'm your AI Study Planner 🤖
  Fatigue-aware · Auto-saves to Google Drive · Plan memory

  ℹ️  No saved plans found — let's create your first one!


──────────────────────────────────────────────────────────
👤  What's your name? pooja

  Nice to meet you, pooja! 🎉
📅  Today's date (e.g. 27 June 2026): 27 june 2026

😴  How TIRED are you right now?
    1  = Wide awake and energised
    5  = Somewhat tired but manageable
    7  = Noticeably tired, hard to focus
    10 = Exhausted, barely keeping eyes open

    Your fatigue level (1–10): 5

  Mode selected →  🟡 Moderate Mode
  Slightly tired — shorter sessions with more rest built in.

⏰  What time do you want to START? (24-hour format)
    e.g. 9 for 9 AM, 14 for 2 PM
    Start hour (0–23): 9 AM
    ⚠️  Please enter a number.
    Start hour (0–23): 9

📚  How ma